# Flexible Job Shop Scheduling Problem with Transportation (FJSPT)

Задача FJSPT определяется следующими компонентами:
* Задания: $J = \{1, \dots, n\}$ - набор работ (jobs), $i$-я работа состоит из последовательности (маршрута) $n_i$ операций.
* Станки: $M = \{M_1, \dots, M_m\}$ - набор доступного производственного оборудования (machines).
* Транспорт: $V = \{V_1, \dots, V_{n_V}\}$ - набор транспортных средств (trucks).
* Операции: множество $O$ включает производственные операции $u$ (выполнение операци на станке) и транспортные операции $t_u$ (перемещение детали к месту выполнения производственной операции $u$).

Особенности задачи FJSPT:
* Каждая производственная операция может быть выполнена на любом станке из подмножества допустимых станков $M_u \subseteq M$. Время обработки $p_u$ зависит от выбранного станка.
* Любое ТС может выполнить любую транспортную операцию ($V^{t_u} = V$). Время перевозки $p_{t_u}$ зависит только от расстояния и не зависит от выбора ТС.
* Каждой производственной операции $u$ предшествует транспортировка $t_u$ (включая первую операцию - из депо LU к первому станку). Транспортировка после последней операции не учитывается.
* После начала операция не может быть прервана.
* Каждый станок и транспортное средство в любой момент времени может обрабатывать не более одной операции. 
* Операции одной работы $j$ должны выполняться строго в заданном порядке. Операция $u + 1$ не может начаться, пока не завершится операция $u$ и последующая транспортировка $t_{u+1}$.
* Если две последовательные производственные операции одной работы выполняются на одном и том же станке, транспортная операция между ними отсутствует.
* Время в пути между станками $m_a$ и $m_b$ обозначается как $d(m_a, m_b)$. Транспортные времена удовлетворяют неравенству треугольника: $d(m_1, m_2) \leqslant d(m_1, m_3) + d(m_3, m_2)$.
* Время занятости ТС $p_{t_u}$ складывается из времени перевозки груза и времени порожнего пробега до следующего места погрузки.

Цель задачи FJSPT: минимизация общей продолжительности выполнения всех работ $C_{max}$ (makespan).

## Обучение модели для FJSPT с использованием подхода RL4CO

In [1]:
# from pytorch_lightning.loggers import CometLogger
import torch

from lightning.pytorch.callbacks import ModelCheckpoint, RichModelSummary

from env import FJSPTEnv
from rl4co.utils.trainer import RL4COTrainer

from model import L2DModel
from l2d_policy import L2DPolicy

In [2]:
from rl4co.envs import ENV_REGISTRY

ENV_REGISTRY["fjspt"] = FJSPTEnv

env = FJSPTEnv(
    generator_params={
      "num_jobs": 15,  # the total number of jobs
      "num_machines": 10,  # the total number of machines that can process operations
      "num_trucks": 2,  # the total number of trucks
      "min_ops_per_job": 5,  # minimum number of operatios per job
      "max_ops_per_job": 15,  # maximum number of operations per job
      "min_processing_time": 20,  # the minimum time required for a machine to process an operation
      "max_processing_time": 70,  # the maximum time required for a machine to process an operation
      "min_transportation_time": 0,  # the minimum time required for a truck to transport
      "max_transportation_time": 50,  # the maximum time required for a truck to transport
      "min_eligible_ma_per_op": 1,  # the minimum number of machines capable to process an operation
      "max_eligible_ma_per_op": 5,  # the maximum number of machines capable to process an operation
    },
)

In [3]:
accelerator = "gpu"
batch_size = 128
max_epochs = 100
train_data_size = batch_size * 80
val_data_size = 1_000
test_data_size = 1_000
embed_dim = 128
num_encoder_layers = 4

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = L2DPolicy(embed_dim=embed_dim, num_encoder_layers=num_encoder_layers)
policy = policy.to(device)

model = L2DModel(
    env,
    policy=policy, 
    baseline="rollout",
    batch_size=batch_size,
    train_data_size=train_data_size,
    val_data_size=val_data_size,
    test_data_size=test_data_size,
    optimizer_kwargs={"lr": 1e-4}
)

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


## Обучение

In [5]:
policy.train()

L2DPolicy(
  (encoder): HetGNNEncoder(
    (init_embedding): FJSPTInitEmbedding(
      (init_ops_embed): Linear(in_features=5, out_features=128, bias=False)
      (pos_encoder): PositionalEncoding(
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (init_ma_embed): Linear(in_features=1, out_features=128, bias=False)
      (init_truck_embed): Linear(in_features=1, out_features=128, bias=False)
      (proc_edge_embed): Linear(in_features=1, out_features=128, bias=False)
      (truck_edge_embed): Linear(in_features=1, out_features=128, bias=False)
    )
    (layers): ModuleList(
      (0-3): 4 x HetGNNBlock(
        (ma_from_ops): HetGNNLayer(
          (activation): ReLU()
        )
        (ops_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (ma_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (truck_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (ffn_ma): TransformerFFN(
          (ops): ModuleDict(
   

In [6]:
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints_reinforce",
    filename="reinforce_{epoch:03d}",
    save_top_k=1,
    save_last=True,
    monitor="val/reward",
    mode="max",  # maximize reward = minimize makespan
)

rich_model_summary = RichModelSummary(max_depth=3)

callbacks = [checkpoint_callback, rich_model_summary]

In [7]:
# from comet_key import API_KEY

# logger = CometLogger(
#     api_key=API_KEY,
#     project="rl4co_reinforce",
# )

trainer = RL4COTrainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=1,
    logger=None,  # logger,
    callbacks=callbacks,
)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/fsoidhfoi/rl4co-reinforce2/b8caeac933074f6d9b61bf6c22decd04

Using 16bit Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model)

val_file not set. Generating dataset instead
test_file not set. Generating dataset instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                          ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ env                           │ FJSPTEnv            │      0 │ train │     0 │
│ 1  │ policy                        │ L2DPolicy           │  870 K │ train │     0 │
│ 2  │ policy.encoder                │ HetGNNEncoder       │  804 K │ train │     0 │
│ 3  │ policy.encoder.init_embedding │ FJSPTInitEmbedding  │  1.2 K │ train │     0 │
│ 4  │ policy.encoder.layers         │ ModuleList          │  803 K │ train │     0 │
│ 5  │ policy.decoder                │ L2DDecoder          │ 66.3 K │ train │     0 │
│ 6  │ policy.decoder.actor          │ FJSPTActor          │ 66.3 K │ train │     0 │
│ 7  │ baseline                      │ WarmupBaseline      │  870 K │ train │     0 │
│ 8  │ baseline.baseline             │ RolloutBaseline     │  870 K │ train │     0 │
│ 9  │ baseline.baseline.policy      │ L2DPolicy           │  870 K │ eval  │     0 │
│ 10 │ baseline.warmup_baseline      │ ExponentialBaseline │      0 │ train │     0 │
└────┴───────────────────────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 182                                                                                         
Modules in eval mode: 178                                                                                          
Total FLOPs: 0

Output()

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connect
or.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value
of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connect
or.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:534: Found 
178 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this 
is intentional, you can ignore this warning.

## Тестирование обученной модели

In [1]:
import torch
from env import FJSPTEnv
from model import L2DModel
from pathlib import Path
import pandas as pd
from rl4co.envs import ENV_REGISTRY

ENV_REGISTRY["fjspt"] = FJSPTEnv

datasets_params = [
    ("1_Deroussi_and_Norre", "datasets/1_Deroussi_and_Norre/proc_data", 1),
    ("2_Ham_edata", "datasets/2_Ham/edata", 1),
    ("2_Ham_rdata", "datasets/2_Ham/rdata", 1),
    ("2_Ham_sdata", "datasets/2_Ham/sdata", 1),
    ("2_Ham_vdata", "datasets/2_Ham/vdata", 1),
    ("3_Homayouni_and_Fontes-Brandimarte", "datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data", 1),
    ("4_Homayouni_and_Fontes-Fattahi", "datasets/4_Homayouni_and_Fontes-Fattahi/proc_data", 0),
    ("5_Dauzere", "datasets/5_Dauzere/proc_data", 1),
]

results = {}

model_checkpoint = L2DModel.load_from_checkpoint(
    "checkpoints_reinforce/reinforce_epoch=089.ckpt", 
    strict=False,
    weights_only=False
)
device = torch.device("cpu")
policy = model_checkpoint.policy.to(device).eval()

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['baseline.baseline.policy.encoder.init_embedding.init_ops_embed.weight', 'baseline.baseline.policy.encoder.init_embedding.pos_encoder.pe', 'baseline.baseline.policy.encoder.init_embedding.init_ma_embed.weight', 'baseline.baseline.policy.encoder.init_em

#### Сохранение результатов

In [2]:
for dataset_name, proc_path, ma_indexing in datasets_params:
    print("-" * 20)
    print(proc_path)
    path = Path(proc_path)
    rows = []
    
    for file in path.iterdir():
        if not file.is_file():
            continue
        print(file)
  
        # Окружение для тестирования модели с помощью подготовленных датасетов
        env = FJSPTEnv(
            generator_params={
              # датасеты с данными FJSP
              "proc_file_path": file,
              # датасет с временем транспортировки
              "trucks_file_path": Path(proc_path).parent / "trucks_data",
              "ma_indexing": ma_indexing,
            },
        )
        td = env.reset(batch_size=[1]).to(device)
        out = policy(td, env, phase="test", decode_type="greedy")
        rows.append({
            "file": str(file).split("/")[-1],
            "reinforce_policy": -out["reward"].item()
        })

    df = pd.DataFrame(rows).sort_values("file")
    save_path = f"reinforce_results/{dataset_name}.csv"
    df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

--------------------
../datasets/1_Deroussi_and_Norre/proc_data
../datasets/1_Deroussi_and_Norre/proc_data/fjsp5.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp2.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp8.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp6.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp9.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp10.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp3.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp7.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp4.txt
../datasets/1_Deroussi_and_Norre/proc_data/fjsp1.txt
Saved to reinforce_results/1_Deroussi_and_Norre.csv
--------------------
../datasets/2_Ham/edata
../datasets/2_Ham/edata/la06.fjs
../datasets/2_Ham/edata/abz5.fjs
../datasets/2_Ham/edata/la01.fjs
../datasets/2_Ham/edata/orb7.fjs
../datasets/2_Ham/edata/mt06.fjs
../datasets/2_Ham/edata/la13.fjs
../datasets/2_Ham/edata/car7.fjs
../datasets/2_Ham/edata/la26.fjs
../datasets/2_Ham/edata/abz7.fjs
../datasets

Усреднение makespan $С_{max}$ для датасета Ham

In [4]:
import pandas as pd

files = ["edata", "rdata", "sdata", "vdata"]
results = []
for HUdata in files:
    df = pd.read_csv(f"reinforce_results/2_Ham_{HUdata}.csv", sep=',', index_col=0)
    means = df[["reinforce_policy"]].mean().round(1)
    means["HUdata"] = HUdata
    means = means[["HUdata", "reinforce_policy"]]
    results.append(means)


result_df = pd.DataFrame(results)
result_df.to_csv("reinforce_results/2_Ham.csv", index=False)
result_df

,HUdata,reinforce_policy
0,edata,6855.7
1,rdata,6016.9
2,sdata,7260.4
3,vdata,5623.9
